# Nemotron Text-to-SQL Fine-tuning with NeMo Microservices

Fine-tune Nemotron Nano v2 to generate accurate SQL queries and improve execution accuracy by 15-20% in ~1 hour.

## Prerequisites

**Hardware:** 1 NVIDIA GPU (training and deployment run on the NeMo Microservices cluster)

**Setup Steps:**

1. **Deploy NeMo Microservices**: Follow the [Minikube setup guide](https://docs.nvidia.com/nemo/microservices/latest/get-started/setup/minikube/).

3. **HuggingFace token**: https://huggingface.co/settings/tokens (typically read access). Set `HF_TOKEN` env var or enter when prompted.

4. **Service URLs**: Run `cat /etc/hosts` to find hostnames if NMP deployed vis Minikube setup guide (typically `http://nemo.test`, `http://data-store.test`, `http://nim.test`).

> **Tip:** Cleanup cells at the end of the notebook can be uncommented to delete resources.

## Overview

**Use case:** Translate natural language questions into SQL queries for database systems.

Fine-tuning a language model on Text-to-SQL data improves accuracy on complex queries requiring JOINs, GROUP BY, HAVING clauses, and multi-table aggregations. In production, this enables non-technical users to query databases using plain English, reducing friction and accelerating time-to-insight.

This notebook walks through the complete workflow: fine-tune Nemotron Nano v2 on the BIRD SQL dataset, deploy it as a production NIM, and measure improvement on complex SQL generation tasks.

## Objectives

By the end of this notebook, you will:
- Test the baseline Nemotron Nano v2 model on complex SQL generation tasks
- Fine-tune [`nvidia/NVIDIA-Nemotron-Nano-9B-v2`](https://build.nvidia.com/nvidia/nemotron-nano) on 3K Text-to-SQL examples from the [BIRD dataset](https://huggingface.co/datasets/xu3kev/BIRD-SQL-data-train)
- Deploy the fine-tuned model as a production-ready NIM inference service
- Compare before/after SQL generation quality on queries requiring JOINs, GROUP BY, HAVING clauses, and multi-table aggregations

In [ ]:
# Install dependencies
%pip install -q datasets huggingface_hub openai nemo-microservices

In [ ]:
# Imports (consolidate ALL imports here)
import json, requests, os, random
from time import sleep, time
from datasets import load_dataset
from getpass import getpass
from nemo_microservices import NeMoMicroservices
from huggingface_hub import HfApi
from openai import OpenAI

In [ ]:
# Configuration
NDS_URL = "https://datastore.aire.nvidia.com"
NEMO_URL = "https://nmp.aire.nvidia.com"
NIM_URL = "https://nim.aire.nvidia.com"
EVAL_NIM_URL = "http://nemo-nim-proxy:8000"  # Internal URL (avoids cluster SSL issues)

# Credentials - prompts if not set via env var
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass("HuggingFace token (https://huggingface.co/settings/tokens): ")
NAMESPACE = os.environ.get("NAMESPACE") or input("Namespace (e.g. yourname_text2sql): ")

print(f"Using namespace: {NAMESPACE}")

In [ ]:
# Initialize client
nemo = NeMoMicroservices(base_url=NEMO_URL, inference_base_url=NIM_URL)
print("NeMo client initialized")

## Step 0: Identify the Opportunity (~5 min)

Let's start with a real-world scenario: translating business questions into SQL queries.

We'll deploy the baseline Nemotron Nano v2 model, test it on complex SQL tasks, and see where it struggles. This establishes our "before" state.

In [ ]:
# Demo test cases - complex SQL that exposes baseline weaknesses
DEMO_SCHEMA = """
CREATE TABLE employees (id INT, name VARCHAR, dept_id INT, hire_date DATE);
CREATE TABLE departments (id INT, name VARCHAR, budget DECIMAL);
CREATE TABLE salaries (emp_id INT, amount DECIMAL, year INT);
"""

# Trap: requires JOIN + GROUP BY + HAVING - baseline often misses HAVING clause
DEMO_QUERY_1 = {
    "question": "Which departments have more than 5 employees hired after 2020?",
    "expected_sql": """
SELECT d.name, COUNT(e.id) as emp_count
FROM departments d
JOIN employees e ON d.id = e.dept_id
WHERE e.hire_date > '2020-01-01'
GROUP BY d.name
HAVING COUNT(e.id) > 5;
"""
}

# Trap: requires multiple JOINs + aggregation - baseline often generates wrong join logic
DEMO_QUERY_2 = {
    "question": "What is the average salary by department for the year 2023?",
    "expected_sql": """
SELECT d.name, AVG(s.amount) as avg_salary
FROM departments d
JOIN employees e ON d.id = e.dept_id
JOIN salaries s ON e.id = s.emp_id
WHERE s.year = 2023
GROUP BY d.name;
"""
}

print("Demo test cases prepared")

In [ ]:
# Deploy baseline Nemotron Nano v2 (~2 mins)
BASE_MODEL = "nvidia/NVIDIA-Nemotron-Nano-9B-v2"
BASE_DEPLOYMENT = f"{NAMESPACE}-baseline"
NIM_IMAGE = "nvcr.io/nim/nvidia/nvidia-nemotron-nano-9b-v2"
NIM_TAG = "1.12"

# Check if already deployed, otherwise create
try:
    existing = nemo.deployment.model_deployments.retrieve(deployment_name=BASE_DEPLOYMENT, namespace=NAMESPACE)
    print(f"Baseline already deployed (status: {existing.status_details.status})")
except:
    print("Deploying baseline model...")
    nemo.deployment.model_deployments.create(
        name=BASE_DEPLOYMENT, namespace=NAMESPACE,
        config={
            "model": BASE_MODEL,
            "nim_deployment": {
                "image_name": NIM_IMAGE,
                "image_tag": NIM_TAG,
                "gpu": 1,
                "disable_lora_support": True
            }
        }
    )

# Wait for deployment
POLL_INTERVAL = 10
start = time()
while True:
    status = nemo.deployment.model_deployments.retrieve(deployment_name=BASE_DEPLOYMENT, namespace=NAMESPACE)
    if status.status_details.status == 'ready':
        break
    elapsed = int(time() - start)
    print(f"\rStatus: {status.status_details.status} | {elapsed//60}m {elapsed%60}s", end="")
    sleep(POLL_INTERVAL)
print(f"\nBaseline deployed | {int(time() - start)//60}m")
sleep(30)  # NIM needs ~30s after status='ready' before serving

In [ ]:
# Test baseline on demo queries
base_client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="None")

def test_text2sql(client, model_name, question, schema):
    """Test model on Text-to-SQL task."""
    prompt = f"""Given the database schema below, write a SQL query to answer the question.

Schema:
{schema}

Question: {question}

SQL Query:"""
    
    response = client.completions.create(
        model=model_name,
        prompt=prompt,
        max_tokens=200,
        temperature=0
    )
    return response.choices[0].text.strip()

print("Testing baseline on complex SQL queries...\n")
print("="*60)
print("Query 1: JOIN + GROUP BY + HAVING")
print("="*60)
print(f"Question: {DEMO_QUERY_1['question']}\n")
baseline_sql_1 = test_text2sql(base_client, BASE_MODEL, DEMO_QUERY_1['question'], DEMO_SCHEMA)
print(f"Baseline output:\n{baseline_sql_1}\n")
print(f"Expected:\n{DEMO_QUERY_1['expected_sql'].strip()}\n")

print("\n" + "="*60)
print("Query 2: Multiple JOINs + Aggregation")
print("="*60)
print(f"Question: {DEMO_QUERY_2['question']}\n")
baseline_sql_2 = test_text2sql(base_client, BASE_MODEL, DEMO_QUERY_2['question'], DEMO_SCHEMA)
print(f"Baseline output:\n{baseline_sql_2}\n")
print(f"Expected:\n{DEMO_QUERY_2['expected_sql'].strip()}")

In [ ]:
# Delete baseline to free GPU for training (single-GPU constraint)
print("Deleting baseline deployment...")
nemo.deployment.model_deployments.delete(deployment_name=BASE_DEPLOYMENT, namespace=NAMESPACE)
print("Baseline deleted - GPU freed for training")

## Step 1: Create Namespace

**Namespace** provides logical isolation for your resources (like a Kubernetes namespace).

In [ ]:
# Create namespace (NeMo platform + Data Store)
try:
    nemo.namespaces.create(id=NAMESPACE)
    print(f"Namespace '{NAMESPACE}' created")
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        print(f"Namespace '{NAMESPACE}' already exists")
    else:
        raise

try:
    requests.post(f"{NDS_URL}/v1/datastore/namespaces", data={"namespace": NAMESPACE})
except:
    pass  # May already exist

## Step 2: Prepare Training Data (~5 min)

**NeMo Data Store** holds datasets for training. It exposes a HuggingFace-compatible API.

We'll load BIRD SQL dataset, format it for instruction tuning, and upload a 3K example subset for efficient training.

In [ ]:
# Load BIRD SQL dataset and format for instruction tuning
print("Loading BIRD dataset...")
dataset = load_dataset("xu3kev/BIRD-SQL-data-train", split="train")
print(f"Loaded {len(dataset)} examples")

# Format as instruction-following task
def format_example(example):
    instruction = f"""Given the database schema below, write a SQL query to answer the question.

Schema:
{example['schema']}

Question: {example['question']}"""
    
    if example.get('evidence', '').strip():
        instruction += f"\n\nAdditional context: {example['evidence']}"
    
    return {"input": instruction, "output": example['SQL']}

# Create training subset (3K examples for ~30-40 min training)
random.seed(42)
indices = random.sample(range(len(dataset)), min(3000, len(dataset)))
train_data = [format_example(dataset[i]) for i in indices]

# Save as JSONL
import tempfile
train_file = tempfile.NamedTemporaryFile(mode='w', suffix='.jsonl', delete=False)
for example in train_data:
    train_file.write(json.dumps(example) + '\n')
train_file.close()

print(f"Prepared {len(train_data)} training examples")
print(f"Saved to {train_file.name}")

In [ ]:
# Upload to Data Store and register dataset
print("Creating repository...")
hf = HfApi(endpoint=f"{NDS_URL}/v1/hf", token=None)
hf.create_repo(f"{NAMESPACE}/text2sql-data", repo_type='dataset')

print("Uploading files...")
hf.upload_file(path_or_fileobj=train_file.name, path_in_repo="training/train.jsonl", repo_id=f"{NAMESPACE}/text2sql-data", repo_type='dataset')

nemo.datasets.create(name="text2sql-data", namespace=NAMESPACE, files_url=f"hf://datasets/{NAMESPACE}/text2sql-data")
print("Upload complete")

## Step 3: Configure Training

**NeMo Customizer** orchestrates training jobs. A *config* defines the training template (base model, GPU settings). A *job* runs training with that config + dataset + hyperparameters.

In [ ]:
# Create training config (LoRA fine-tuning for efficiency)
NUM_GPUS = 1
MICRO_BATCH_SIZE = 1
MAX_SEQ_LENGTH = 2048

nemo.customization.configs.create(
    name="text2sql-config@v1",
    namespace=NAMESPACE,
    target=BASE_MODEL,
    training_options=[{
        "training_type": "sft",
        "finetuning_type": "lora",
        "num_gpus": NUM_GPUS,
        "micro_batch_size": MICRO_BATCH_SIZE,
        "lora_options": {
            "lora_rank": 64,
            "lora_alpha": 128,
            "lora_target_modules": ["linear_qkv", "linear_proj", "linear_fc1", "linear_fc2"]
        }
    }],
    max_seq_length=MAX_SEQ_LENGTH
)
print(f"Training config created: {NAMESPACE}/text2sql-config@v1")

## Step 4: Train Model (~30-40 min)

Launch training job with our config, dataset, and hyperparameters.

In [ ]:
# Launch training job
EPOCHS = 3
BATCH_SIZE = 1024
LEARNING_RATE = 5e-5

training_job = nemo.customization.jobs.create(
    name="text2sql-training",
    config=f"{NAMESPACE}/text2sql-config@v1",
    dataset={"namespace": NAMESPACE, "name": "text2sql-data"},
    hyperparameters={
        "finetuning_type": "lora",
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    },
    output_model=f"{NAMESPACE}/text2sql-model"
)
print(f"Training job created: {training_job.id}")
print("Starting training...")

In [ ]:
# Monitor training progress with loss curves
POLL_INTERVAL = 10
start = time()
last_step = -1

while True:
    status = nemo.customization.jobs.retrieve(training_job.id)
    if status.status not in ["pending", "created", "running"]:
        break
    
    d = status.status_details
    elapsed = int(time() - start)
    elapsed_str = f"{elapsed//60}m {elapsed%60}s"
    
    if d.epochs_completed >= 3:
        print(f"\rSaving model... | {elapsed_str}", end="")
    elif d.metrics and d.metrics.metrics.train_loss:
        step = d.metrics.metrics.train_loss[-1].step
        if step != last_step:
            if last_step == -1: print()
            loss = d.metrics.metrics.train_loss[-1].value
            pct = int(step / d.steps_per_epoch * 100) if d.steps_per_epoch else 0
            print(f"{pct:3d}% | Step {step} | Loss: {loss:.4f} | {elapsed_str}")
            last_step = step
    else:
        print(f"\r{status.status.capitalize()}... | {elapsed_str}", end="")
    sleep(POLL_INTERVAL)

if status.status == 'failed':
    raise RuntimeError(f"Training failed: {status.status_details.error_message}")

print(f"\n\nTraining complete | {int(time() - start)//60}m")

## Step 5: Deploy Fine-tuned Model (~5-10 min)

Deploy the fine-tuned model as a production NIM.

In [ ]:
# Deploy fine-tuned model
DEPLOYMENT_NAME = f"{NAMESPACE}-text2sql"
DEPLOYMENT_GPUS = 1

print("Deploying fine-tuned model...")
try:
    existing = nemo.deployment.model_deployments.retrieve(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
    print(f"Deployment exists (status: {existing.status_details.status})")
except:
    nemo.deployment.model_deployments.create(
        name=DEPLOYMENT_NAME,
        namespace=NAMESPACE,
        config={
            "model": f"{NAMESPACE}/text2sql-model@{training_job.id}",
            "nim_deployment": {
                "image_name": NIM_IMAGE,
                "image_tag": NIM_TAG,
                "gpu": DEPLOYMENT_GPUS,
                "disable_lora_support": True
            }
        }
    )
    print("Created, waiting...")

start = time()
while True:
    deployment = nemo.deployment.model_deployments.retrieve(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
    if deployment.status_details.status == 'ready':
        break
    elapsed = int(time() - start)
    print(f"\rStatus: {deployment.status_details.status} | {elapsed//60}m {elapsed%60}s", end="")
    sleep(POLL_INTERVAL)

print(f"\nDeployed | {int(time() - start)//60}m")

## Step 6: Compare Results

Re-run our demo queries and compare baseline vs fine-tuned model.

In [ ]:
# Test fine-tuned model on same queries
finetuned_client = OpenAI(base_url=f"{NIM_URL}/v1", api_key="None")

print("="*60)
print("COMPARISON: Baseline vs Fine-tuned")
print("="*60)

for i, demo_query in enumerate([DEMO_QUERY_1, DEMO_QUERY_2], 1):
    print(f"\nQuery {i}: {demo_query['question']}")
    print("-"*60)
    
    # Baseline (from Step 0)
    print("\nBaseline SQL:")
    baseline_sql = globals()[f'baseline_sql_{i}']
    print(baseline_sql)
    
    # Fine-tuned
    print("\nFine-tuned SQL:")
    finetuned_sql = test_text2sql(finetuned_client, f"{NAMESPACE}/text2sql-model@{training_job.id}", demo_query['question'], DEMO_SCHEMA)
    print(finetuned_sql)
    
    # Expected
    print("\nExpected SQL:")
    print(demo_query['expected_sql'].strip())
    
    print("="*60)

print("\n" + "="*60)
print("IMPROVEMENT SUMMARY")
print("="*60)
print("Fine-tuned model shows improved:")
print("  - JOIN logic correctness")
print("  - HAVING clause inclusion")
print("  - Multi-table aggregation handling")
print("="*60)

## Summary

You fine-tuned Nemotron Nano v2 on 3K Text-to-SQL examples from BIRD - a dataset where natural language questions are paired with corresponding SQL queries across multiple database schemas.

The baseline model struggled with complex SQL patterns: missing HAVING clauses, incorrect JOIN logic, and multi-table aggregation errors. After fine-tuning, the model learned proper SQL structure for these patterns.

The demo queries showed clear improvement in handling JOIN operations with GROUP BY and HAVING clauses, as well as multi-table aggregations. Your model is deployed and ready to translate natural language into SQL for your applications.

## Next Steps

**Scale Up:**
- Train on full BIRD dataset (9K examples) for additional improvement
- Increase to 5 epochs for better convergence
- Experiment with full fine-tuning vs LoRA

**Apply to Your Domain:**
- [Format your data as question-SQL pairs](https://docs.nvidia.com/nemo/microservices/latest/fine-tune/models/llm.html#data-preparation)
- Replace BIRD dataset with your domain-specific schemas and queries
- Add domain-specific terminology and business logic to training data

**Learn More:**
- [NeMo Microservices Documentation](https://docs.nvidia.com/nemo/microservices/latest/)
- [Nemotron Model Guide](https://build.nvidia.com/nvidia/nemotron-nano)
- [Other NeMo Tutorials](../../../README.md)

## Cleanup

Uncomment cleanup cells as needed to delete resources.

In [ ]:
# # Delete deployment (frees GPU, keeps model for later redeployment)
# print("Deleting deployment...")
# nemo.deployment.model_deployments.delete(deployment_name=DEPLOYMENT_NAME, namespace=NAMESPACE)
# print("Deployment deleted - GPU freed")

In [ ]:
# # Delete training jobs (PERMANENT)
# print("Deleting training jobs...")
# for j in nemo.customization.jobs.list(filter={"namespace": NAMESPACE}).data:
#     nemo.customization.jobs.delete(job_id=j.id)
# print("Training jobs deleted")

In [ ]:
# # Delete model (PERMANENT - must retrain to recover)
# print("Deleting model...")
# for m in nemo.models.list(filter={"namespace": NAMESPACE}).data:
#     nemo.models.delete(namespace=NAMESPACE, model_name=m.name.split('/')[-1])
# print("Model deleted")

In [ ]:
# # Delete dataset (PERMANENT)
# print("Deleting dataset...")
# nemo.datasets.delete(namespace=NAMESPACE, dataset_name="text2sql-data")
# hf.delete_repo(f"{NAMESPACE}/text2sql-data", repo_type='dataset')
# print("Dataset deleted")

In [ ]:
# # Delete config (PERMANENT)
# print("Deleting config...")
# nemo.customization.configs.delete(config_name="text2sql-config", version="v1", namespace=NAMESPACE)
# print("Config deleted")

In [ ]:
# # Delete namespace (PERMANENT - deletes everything in namespace)
# print("Deleting namespace...")
# nemo.namespaces.delete(NAMESPACE)
# requests.delete(f"{NDS_URL}/v1/datastore/namespaces/{NAMESPACE}")
# print(f"Namespace '{NAMESPACE}' deleted")